<a href="https://colab.research.google.com/github/kny1209/test2/blob/main/AI/DR-08079_%EA%B5%AC%EB%82%98%EC%98%81_AI%EC%9D%91%EC%9A%A9_10%EC%B0%A8%EC%8B%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

문항 1. 정확도와 설명력의 트레이드오프 (Trade-off) 일반적으로 딥러닝 모델(Deep Learning)은 전통적인 머신러닝 모델(Decision Tree, Linear Regression 등)에 비해 예측 성능은 뛰어나지만, 내부 동작 과정을 이해하기 어렵다는 단점이 있습니다. 이처럼 모델이 복잡해질수록 성능은 높아지지만 설명력은 낮아지는 관계를 무엇이라고 합니까? 또한, 이러한 딥러닝 모델을 내부를 알 수 없다는 의미에서 흔히 ( A ) 모델이라고 부릅니다.

답변 작성란:

관계: 정확도와 설명력의 트레이드오프 (Accuracy-Interpretability Trade-off)

( A ): 블랙박스

문항 2. Integrated Gradients의 핵심 아이디어 기존의 Saliency Map(단순 기울기) 방식은 기울기 소실(Gradient Saturation) 문제로 인해 특정 픽셀의 중요도를 0으로 잘못 판단하는 경우가 있습니다. 이를 해결하기 위해 Integrated Gradients는 입력 이미지가 ( A ) 이미지(예: 검은색 이미지)에서 원본 이미지로 서서히 변화할 때의 기울기들을 모두 **( B )** 하여 중요도를 계산합니다.

답변 작성란:

( A ): 기준선(Baseline) / 레퍼런스 (Reference)

( B ): 합산 / 누적 / 적분 (Integral)

문항 3. Saliency Map (단순 기울기) 추출 가장 기초적인 설명 기법인 Saliency Map을 사용하여 입력 이미지의 각 픽셀이 예측 결과에 미치는 영향을 계산해 봅시다. Captum의 Saliency 클래스를 사용하세요.

In [ ]:
from captum.attr import Saliency

# model: 학습된 CNN 모델
# captum_input: 분석할 입력 이미지 (Tensor, requires_grad=True)
# sample_targets: 정답 클래스 인덱스

# TODO: Saliency 객체 생성
saliency = nn.Saliency(model)

# TODO: 어트리뷰션(기여도) 계산
# target에는 정답 클래스 인덱스를 넣어주세요.
gradients = saliency.attribute(captum_input, target=sample_targets[0].item())

print("Gradients shape:", gradients.shape)

문항 4. Integrated Gradients (통합 그래디언트) 적용 Saliency Map의 한계를 극복한 Integrated Gradients 기법을 적용해 봅니다. 기준선(Baseline)으로는 모든 픽셀이 0인(검은색) 이미지를 사용합니다.

In [ ]:
from captum.attr import IntegratedGradients

# TODO: IntegratedGradients 객체 생성
integ_grads = nn.IntegratedGradients(model)

# TODO: 어트리뷰션 계산
# baselines는 입력과 동일한 크기의 0으로 채워진 텐서를 사용하세요. (captum_input * 0)
attributed_ig, delta = integ_grads.attribute(
    captum_input,
    target=sample_targets[0].item(),
    baselines=captum_input * 0,
    return_convergence_delta=True   # 근사 적분 값과 실제 함수 값의 차이(델타) 반환 >> 계산의 정확성 확인 가능
)

문항 5. DeepLIFT (Deep Learning Important FeaTures) 적용 DeepLIFT는 역전파 시 활성화 값의 차이를 전파하여 기여도를 계산하는 방식입니다. Captum을 이용해 DeepLIFT를 적용하는 코드를 작성하세요.

In [ ]:
from captum.attr import DeepLift

# TODO: DeepLift 객체 생성
deep_lift = nn.DeepLift(model)

# TODO: 어트리뷰션 계산
attributed_dl = deep_lift.attribute(
    captum_input,
    target=sample_targets[0].item(),
    baselines=captum_input * 0
)

문항 6. 결과 시각화 (Visualization) 구해진 중요도 맵(Attribution Map)을 원본 이미지 위에 겹쳐서(Overlay) 시각화합니다. captum.attr.visualization 모듈을 사용하세요.

In [ ]:
from captum.attr import visualization as viz
import numpy as np

# 데이터 전처리 (Tensor -> Numpy 변환 및 차원 변경)
# attributed_ig는 (1, 1, 28, 28) 형태라고 가정
attr_ig_np = attributed_ig.squeeze().cpu().detach().numpy()
attr_ig_np = np.reshape(attr_ig_np, (28, 28, 1))

# 원본 이미지 (orig_image) 준비됨

# TODO: 시각화 함수 호출
# method="blended_heat_map", sign="all" (양수/음수 기여도 모두 표시)
_ = viz.visualize_image_attr(
    attr_ig_np,        # 중요도 맵
    orig_image,        # 원본 이미지 (배경)
    method='blended_heat_map',
    sign="all",        # 긍정적 기여(G,R)와 부정적 기여(B)를 모두 표시 >> 모델 판단 근거 입체적으로 보여줌
    show_colorbar=True,
    title="Overlayed Integrated Gradients"
)